## Prompt Engineering
- Prompt Engineering is the art and science of designing and optimizing prompts to guide AI models, particularly Large Language Models (LLMs), towards generating the desired responses.
- Following key-elements helps to design a good prompt:
  - **Context:** Providing relevant background information to help the model understand the task. 
  - **Instructions:** Clearly stating what the model needs to do. 
  - **Examples:** Showing the model how to respond in specific situations. 
  - **Format:** Using appropriate language and structure for the prompt. 
- In this notebook, few types of prompting techniques are illustrated.

In [ ]:
%run /Users/csharm33/code/genai_dbx/Course2/.setup/learner_setup.ipynb

In [ ]:
import openai
import os
import json
import httpx
from dotenv import load_dotenv
from tenacity import (
    retry,
    stop_after_attempt,
    wait_random_exponential,
)

In [ ]:
# Authentication:
def get_access_token():
    auth = "https://api.uhg.com/oauth2/token"
    scope = "https://api.uhg.com/.default"
    grant_type = "client_credentials"


    with httpx.Client() as client:
        body = {
            "grant_type": grant_type,
            "scope": scope,
            "client_id": client_id,
            "client_secret": client_secret,
        }
        headers = {"Content-Type": "application/x-www-form-urlencoded"}
        resp = client.post(auth, headers=headers, data=body, timeout=60)
        access_token = resp.json()["access_token"]
        return access_token


load_dotenv('/Users/csharm33/code/genai_dbx/Course2/Data/vars.env')

endpoint = os.environ.get("MODEL_ENDPOINT")
model_name = os.environ.get("MODEL_NAME")
project_id = os.environ.get("PROJECT_ID")
api_version = os.environ.get("API_VERSION")
client_id = os.environ.get("CLIENT_ID")
client_secret = os.environ.get("CLIENT_SECRET")


chat_client = openai.AzureOpenAI(
        azure_endpoint=endpoint,
        api_version=api_version,
        azure_deployment=model_name,
        azure_ad_token=get_access_token(),
        default_headers={
            "projectId": project_id
        }
    )


In [ ]:
@retry(wait=wait_random_exponential(min=45, max=120), stop=stop_after_attempt(6))
def query_llm(prompt_messages, max_tokens=4096, temperature=1.0, top_p=1.0):

    response = chat_client.chat.completions.create(
        messages=prompt_messages,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        model=model_name
    )

    return {'text': response.choices[0].message.content}

### Zero-shot Prompting
- In this prompting technique, the model performs a task without any prior examples from the user, but instructions alone define the desired output.
- When we know the exact task and able to clearly instruct, then zero-shot prompting can be used. 
- For example, when a ticket has to be routed to a particular department, the model may not require a specific example to do the task, but can automatically route support tickets to the correct department (e.g., Billing, Technical, Sales).

In [ ]:
query = "My invoice for order #1234 seems incorrect. Can you clarify the charges?"
prompt_messages = [
            {
                "role": "developer",
                "content": "Classify this customer query into one of: Billing, Technical, Sales. Respond ONLY with the category name."
            },
            {
                "role": "user",
                "content": query
            }
        ]

response = query_llm(prompt_messages, temperature=0)
print("The said query belongs to " + response['text'] + " Section")

### Few-Shot Prompting
- In this prompting technique, the model is given 3–5 examples to learn the patterns
- When to use: Complex rules, edge cases, domain-specific logic
- Use cases: Compliance checks, contract review
- Consider a below demonstration to detect Compliance Violations in Internal Communications

In [ ]:
query = """ Determine whether a message violates compliance policies.
            Examples:
            Example 1: "input": "Share the customer's credit card details with the marketing team.", "output": "Violation: GDPR"
            Example 2: "input": "Please review the attached contract draft.", "output": "Compliant"
            Example 3: "input": "Terminate John immediately; no need for HR process.", "output": "Violation: HR Policy"

            Now, identify whether the following instruction violates any policy : 
            Forward all patient records to the external consultant.
        """

prompt_messages = [ {"role": "user",  "content": query}]
            
response = query_llm(prompt_messages, temperature=0)
print("The current request may violate: ",response['text'])

## Chain of Thought (CoT) Prompting
- A technique of providing a series of steps in the prompt, so that the model can yield the result by following the steps mentioned in the prompt.
- CoT prompting enables complex reasoning capabilities through intermediate reasoning steps.
- One can combine it with few-shot prompting to get better results on more complex tasks that require reasoning before responding.

***NOTE:***
- CoT Prompting was a necessity during initial language models to get appropriate results. However, the current day LLMs - especially, GPT 3 onwards, have CoT process embedded in their learning. Hence, without giving CoT types of prompts also, they generate accurate answers for most of the complex problems.
- Evolution of the language models and their ability in solving scientific/mathematical problems is given here: https://research.google/blog/evaluating-progress-of-llms-on-scientific-problem-solving/
- Older models like `gpt-2`, `text-davinci-2`, `t5` etc. used to fail to answer mathematical reasoning questions with a normal prompt, but learned from chain of thought type of prompting, if provided by the user. However, these models are now deprecated in current versions of Python and hence cannot be used for illustration.
- In the following example, the style of CoT and non-CoT prompts are shown. However, both may result into similar outcome, because the current day models have CoT process in-built within them.

In [ ]:
# without CoT
query = """ Roger has 5 tennis balls. He buys 2 more cans of tennis balls. 
            Each can has 3 tennis balls. How many tennis balls does he have now?
            
        """

prompt_messages = [ {"role": "user",  "content": query}]
            
response = query_llm(prompt_messages, temperature=0)
print(response['text'] )

In [ ]:
# with CoT
query = """ Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can have 3 tennis balls. 
            How many tennis balls does he have now?
            Answer: Roger has 5 balls initially. 2 cans with 3 each balls means 5 + 2*3 = 11
            Now answer this question:
            John has 5 apples. He buys 2 more crates of apples, and each crate consists of a dozen apple. 
            How many apples does John has now?
        """

prompt_messages = [ {"role": "user",  "content": query}]
            
response = query_llm(prompt_messages, temperature=0)
print(response['text'] )

In [ ]:
# without CoT
query = """ A PPO patient is getting an MRI at an out-of-network imaging center. 
            The plan requires pre-authorization for advanced imaging. 
            Is pre-authorization needed, and who initiates it?
        """
prompt_messages = [ {"role": "user",  "content": query}]
            
response = query_llm(prompt_messages)
print(response['text'] )

In [ ]:
# with CoT
query = """ Let's think through this step by step.

            Scenario:
            - Patient has a PPO plan.
            - Procedure: MRI (advanced imaging).
            - Location: Out-of-network imaging center.
            - The plan requires pre-authorization for advanced imaging.

            Step-by-step:
            1. Does the PPO plan require pre-authorization for advanced imaging? 
            2. Is the MRI considered advanced imaging? 
            3. Does it matter that it's out-of-network? 
            4. Who typically initiates it? 
            5. Give overall conclusion.
        """
prompt_messages = [ {"role": "user",  "content": query}]
            
response = query_llm(prompt_messages)
print(response['text'] )

### Tree of Thought (ToT) Prompting
- Tree of Thought (ToT) prompting is an advanced prompting technique that encourages large language models (LLMs) to explore multiple reasoning paths instead of following a single linear chain of thought.
- Inspired by human decision-making, ToT structures reasoning as a tree, where each node represents a partial solution and branches represent different possible next steps.
- It combines the strengths of deliberation and evaluation, allowing the model to explore, backtrack, and select the most promising path, enhancing accuracy and coherence.
- ToT prompting is particularly useful for complex problem-solving tasks such as math word problems, reasoning puzzles, coding, and multi-step decision-making.
- By simulating internal deliberation, Tree of Thought prompting improves performance over standard chain-of-thought methods, especially when reasoning requires trial-and-error or comparison of alternatives.

In [ ]:
query = """ You are a health insurance advisor evaluating the best plan for a patient.
            Question: Should a patient with chronic diabetes and hypertension be offered a standard health plan or a specialized chronic care plan?

            Think in multiple ways (Tree of Thought):
            1. Think based on cost-effectiveness.
            2. Think based on patient health outcomes and care coordination.
            3. Think based on long-term insurance risk and sustainability.

            Evaluate each path and provide a final recommendation with reasoning.
            Display a tree-structure with proper blocks to show the paths.
        """
prompt_messages = [ {"role": "user",  "content": query}]
            
response = query_llm(prompt_messages)
print(response['text'] )


## Stateful Communication
- Stateful communication means, designing the prompts in such a way taht the model maintains the awareness of past interaction and context.
- This allows coherent, personalized and context-sensitive responses.
- Stateful communication is also referred to as **multi-turn conversation.**
- Designing the prompts that maintain stateful communication is very essential is developing bots.
- Stateful communication can be achieved in two ways:
  - **Message History:** Most chat APIs require you to send prior messages along with the new one to maintain state.
  - **Longterm Memory:** Some systems use external memory like ChatGPT's memory feature, Custom app memory in LangChain or RAG systems.
- Let us see how to design the prompts with stateful communication using *message history*. 

In [ ]:
# Initialize the conversation history
prompt_messages = [
    {"role": "developer", "content": "You are a helpful assistant specialized in health insurance. Be clear and concise."}
]

def chat(user_input):
    # Add user message to conversation history
    prompt_messages.append({"role": "user", "content": user_input})
    
    # Get response from OpenAI
    query_llm(prompt_messages)

    # Extract assistant reply
    assistant_reply = response['text'] 
    
    # Add assistant reply to conversation history
    prompt_messages.append({"role": "assistant", "content": assistant_reply})
    
    return assistant_reply

# Simulate a multi-turn conversation
print("User: What does my health insurance cover?")
print("Assistant:", chat("What does my health insurance cover?"))

print("\nUser: What about dental procedures?")
print("Assistant:", chat("What about dental procedures?"))

print("\nUser: Do I need prior approval for a root canal?")
print("Assistant:", chat("Do I need prior approval for a root canal?"))


##### Comments on model output:
- Observe the initial instruction given to the model through `developer` role.
- The function `chat()` does the following tasks:
  - Received the user inputs or the subsequent queries
  - Calls `query_llm()` to get the reply for the user inputs
  - Appends the reply to `prompt_messages` to keep track of the message history
- To simulate stateful communication, start providing the user inputs. Note that, here we are giving fixed queries (3 in total) as user inputs. However, one can try receiving user queries through key-board inputs (using `input()` function). The real-time bots are designed to receive user queries dynamically and respond accordingly. 